In [1]:
## remove commma 

In [2]:
import pandas as pd
import re
from ftfy import fix_text 
from sentence_transformers import SentenceTransformer

In [3]:
df1 = pd.read_csv('datasets/gpt_dataset.csv')

In [4]:
df2 = pd.read_csv('datasets/resume_dataset.csv')

In [5]:
df3 = pd.read_csv('datasets/Resume.csv')
df3 = df3.rename(columns={'Resume_str':'Resume'})
df3 = df3[['Category','Resume']]


In [6]:
df4 = pd.read_csv('datasets/Preprocessed_Data.csv')
df4 = df4.rename(columns={'Text': 'Resume'})

In [7]:
df = pd.concat([df1, df2, df3, df4])

In [8]:
category_map = {

    # Case differences
    "ACCOUNTANT": "Accountant",
    "ADVOCATE": "Advocate",
    "ARTS": "Arts",
    "AUTOMOBILE": "Automobile",
    "AVIATION": "Aviation",
    "BANKING": "Banking",
    "CONSULTANT": "Consultant",
    "DIGITAL-MEDIA": "Digital Media",
    "FINANCE": "Finance",
    "HEALTHCARE": "Healthcare",
    "PUBLIC-RELATIONS": "Public Relations",
    "DESIGNER": "Designer",
    "ENGINEERING": "Engineering",
    "SALES": "Sales",
    "APPAREL": "Apparel",
    "TEACHER": "Education",
    "HR": "Human Resources",

    # Similar meanings
    "FITNESS": "Health and Fitness",
    "Health and fitness": "Health and Fitness",

    "INFORMATION-TECHNOLOGY": "Information Technology",

    "BUSINESS-DEVELOPMENT": "Business Development",

    "CONSTRUCTION": "Building and Construction",

    "AGRICULTURE": "Agriculture",

    # Merge similar AI roles
    "Data Scientist": "Data Science",

    # Merge DevOps naming
    "DevOps Engineer": "DevOps"
}

df["Category"] = df["Category"].replace(category_map)

In [9]:
df["Resume"] = df["Resume"].apply(fix_text)

In [10]:
counts = df["Category"].value_counts()

valid_categories = counts[counts >= 20].index

df = df[df["Category"].isin(valid_categories)]

In [11]:
df.shape

(16428, 2)

In [12]:
df.head()

,Category,Resume
0,Frontend Developer,"As a seasoned Frontend Developer, I have a pro..."
1,Backend Developer,With a solid background in Backend Development...
2,Python Developer,"As a Python Developer, I leverage my expertise..."
3,Data Science,"With a background in Data Science, I possess a..."
4,Frontend Developer,Experienced Frontend Developer with a passion ...


In [13]:
def clean_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)

    # # Keep letters, numbers, +, #, ., -, and spaces
    text = re.sub(r"[^a-zA-Z0-9+#.\-\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    #remove numbers
    text = re.sub(r"\+?\d[\d\s()-]{8,}\d", " ", text)

    return text

df['Resume'] = df['Resume'].apply(clean_text)

In [14]:
df.head()

,Category,Resume
0,Frontend Developer,as a seasoned frontend developer i have a prov...
1,Backend Developer,with a solid background in backend development...
2,Python Developer,as a python developer i leverage my expertise ...
3,Data Science,with a background in data science i possess a ...
4,Frontend Developer,experienced frontend developer with a passion ...


In [15]:
df.isnull().sum()

Category    0
Resume      0
dtype: int64

In [16]:
from nltk.tokenize import word_tokenize
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# nltk.download("punkt")
# nltk.download("stopwords")
# nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    #1 tokenize
    tokens = word_tokenize(str(text))
    
    #2 stopword
    tokens = [word for word in tokens if word.lower() not in stop_words]
    #3 Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens
    
df['Resume'] = df['Resume'].apply(preprocess)
df["Resume"] = df["Resume"].apply(lambda x: " ".join(x))

In [17]:
df.head()

,Category,Resume
0,Frontend Developer,seasoned frontend developer proven track recor...
1,Backend Developer,solid background backend development bring 7 y...
2,Python Developer,python developer leverage expertise python pro...
3,Data Science,background data science possess unique blend a...
4,Frontend Developer,experienced frontend developer passion craftin...


In [18]:
model = SentenceTransformer("all-mpnet-base-v2")
resume_embedding = model.encode(
    df['Resume'].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/514 [00:00<?, ?it/s]

In [19]:
temp_resume_embedding = resume_embedding

In [20]:
print(temp_resume_embedding.shape)

(16428, 768)


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X = temp_resume_embedding     # Features (after text vectorization)
y = df["Category"]    # Target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [22]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.7279367011564212


In [23]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000,class_weight="balanced")
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7215459525258673
                                    precision    recall  f1-score   support

                        Accountant       0.77      0.94      0.85        94
                          Advocate       0.59      0.64      0.62        84
                       Agriculture       0.78      0.56      0.66        71
                           Apparel       0.70      0.69      0.69        83
                      Architecture       0.86      0.45      0.59        69
                              Arts       0.67      0.44      0.53        88
                        Automobile       0.69      0.53      0.60        70
                          Aviation       0.80      0.81      0.80        91
                               BPO       0.47      0.53      0.50        45
                 Backend Developer       1.00      1.00      1.00        11
                           Banking       0.79      0.73      0.76        86
                        Blockchain       0.91      1.00   

In [24]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
X_train_scale = ss.fit_transform(X_train)
X_test_scale = ss.transform(X_test)

from sklearn.svm import SVC
svc = SVC(class_weight='balanced')
svc.fit(X_train, y_train)

y_pred = svc.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7410225197808886
                                    precision    recall  f1-score   support

                        Accountant       0.79      0.93      0.85        94
                          Advocate       0.55      0.64      0.59        84
                       Agriculture       0.75      0.56      0.65        71
                           Apparel       0.64      0.71      0.67        83
                      Architecture       0.79      0.59      0.68        69
                              Arts       0.68      0.60      0.64        88
                        Automobile       0.56      0.53      0.54        70
                          Aviation       0.82      0.85      0.83        91
                               BPO       0.50      0.53      0.52        45
                 Backend Developer       1.00      1.00      1.00        11
                           Banking       0.81      0.79      0.80        86
                        Blockchain       0.91      1.00   

In [25]:
from sklearn.svm import LinearSVC

linearsvc = LinearSVC(max_iter=5000,class_weight="balanced")
linearsvc.fit(X_train, y_train)
pred = linearsvc.predict(X_test)

print(accuracy_score(y_test, pred))

0.7620206938527084


In [26]:
import joblib

joblib.dump(linearsvc, "models/resume_classifier.pkl")

['models/resume_classifier.pkl']

In [27]:
# import PyMuPDF
import fitz


def pdf_text(pdf_path):
    text = ""
    
    if pdf_path is not None:
        doc = fitz.open(pdf_path)

        for page in doc:
            text += page.get_text()

        doc.close()
    return text

resume_text = pdf_text('Resume.pdf')
clean_resume = clean_text(resume_text)
preprocess_resume = preprocess(clean_resume)
resume_text =  " ".join(preprocess_resume)
resume_embed = model.encode(resume_text,
    convert_to_numpy=True,
    show_progress_bar=True)
print(linearsvc.predict([resume_embed]))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

['Data Science']


In [28]:
import joblib
from sentence_transformers import SentenceTransformer

# Load embedding model
embedding_model = SentenceTransformer("all-mpnet-base-v2")

# Load trained classifier
classifier = joblib.load("models/resume_classifier.pkl")

resume = """
Sarah Williams

Professional Summary

Python Developer with 3 years of experience developing REST APIs,
automation scripts, and backend applications.

Skills

Python
Flask
Django
FastAPI
SQL
PostgreSQL
Git
Docker
REST API

Projects

E-commerce Backend
Attendance Management System
Inventory Management System

Experience

Built backend services using Flask and Django.
Integrated REST APIs and optimized database queries.
"""



clean_resume = clean_text(resume)
processed_resume = preprocess(clean_resume)


processed_resume = " ".join(processed_resume)
resume_embedding = embedding_model.encode(
    [processed_resume],
    convert_to_numpy=True
)

prediction = classifier.predict(resume_embedding)

print("=" * 40)
print("Predicted Category :", prediction[0])
print("=" * 40)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Predicted Category : Python Developer


## ATS SCORE

In [29]:
model = SentenceTransformer("all-mpnet-base-v2")

job_description ="""
AI ML Engineer Job Description Template
The AI ML Engineer is responsible for leveraging big data and machine learning techniques to build advanced analytics models that provide actionable insights and drive continuous improvement in our products and services. You will collaborate with cross-functional teams to implement and scale AI/ML solutions effectively.

Post AI ML Engineer Job
Responsibilities
Develop and implement machine learning models and algorithms.
Collaborate with data scientists and engineers to integrate AI solutions into products.
Analyze and process large datasets to extract meaningful insights.
Design and conduct experiments to evaluate model performance.
Deploy and maintain scalable machine learning pipelines.
Stay up-to-date with the latest AI/ML trends and technologies.
Qualifications
Bachelor's or Master's degree in Computer Science, Statistics, Mathematics, or related field.
Experience with machine learning frameworks such as TensorFlow or PyTorch.
Strong coding skills in Python, R, or other programming languages.
Knowledge of big data processing tools like Hadoop, Spark.
Proven ability to design and implement complex algorithms.
Excellent problem-solving and analytical skills.
Skills
Machine Learning
Deep Learning
Python
TensorFlow
PyTorch
Data Analysis
Big Data Technologies
Algorithm Development
Model Evaluation
Data Preprocessing
"""
resume_embedding = model.encode(
    clean_resume,
    convert_to_numpy=True
)

jd_embedding = model.encode(
    job_description,
    convert_to_numpy=True
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [30]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(
    [resume_embedding],
    [jd_embedding]
)[0][0]

In [31]:
ats_score = round(similarity * 100,2)
print(ats_score)

43.97
